In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
from gluonts.dataset.util import to_pandas

from tsfm_lens.dataset import GiftEvalDataset
from tsfm_lens.metrics import compute_metrics
from tsfm_lens.toto.circuitlens import CircuitLensToto
from tsfm_lens.utils import (
    apply_custom_style,
    load_dyst_samples,
    set_seed,
)

In [ ]:
apply_custom_style("../../config/plotting.yaml")

In [ ]:
DEFAULT_COLORS = list(plt.get_cmap("tab20").colors)  # type: ignore
print(f"{len(DEFAULT_COLORS)} colors")

In [ ]:
DEFAULT_CONTEXT_LENGTH = 4096

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
WORK_DIR = os.getenv("WORK", "/work")
DATA_DIR = os.path.join(WORK_DIR, "data")
plot_save_dir = os.path.join("../../figs", "ablations_toto", "examples")
os.makedirs(plot_save_dir, exist_ok=True)

### Load Pipeline

In [ ]:
# Load the pre-trained model
pipeline = CircuitLensToto("Datadog/Toto-Open-Base-1.0", device_map=device)

In [ ]:
num_layers = pipeline.num_layers
num_heads = pipeline.num_heads

print(f"num_layers: {num_layers}, num_heads: {num_heads}")

In [ ]:
# pipeline.model

In [ ]:
model_param_memory_mb = sum(p.numel() * p.element_size() for p in pipeline.model.parameters()) / (1024 * 1024)
print(f"Model parameter memory: {model_param_memory_mb:.2f} MB")

### Load Data

In [ ]:
split_name = "gift-eval"
subsample_interval = 1

if split_name == "base40":
    split_dir = os.path.join(DATA_DIR, split_name)
    system_name = "Thomas"
    system_name_pretty = system_name

    context_length = 512
    context_start_time = 2000
    context_end_time = context_start_time + context_length

    sample_idx = 0
    selected_dim = [0, 1, 2]

    dyst_coords = load_dyst_samples(system_name, split_dir, one_dim_target=False, num_samples=1)
    dyst_coords = torch.tensor(dyst_coords[sample_idx, selected_dim, :])
    print(dyst_coords.shape)
    dyst_coords = dyst_coords[:, ::subsample_interval]
    context = dyst_coords[:, context_start_time:context_end_time]
    print(context.shape)

    prediction_length = 512
    future_vals = dyst_coords[:, context_end_time : context_end_time + prediction_length]
    print(f"future_vals shape: {future_vals.shape}")

elif split_name == "gift-eval":
    from itertools import islice

    split_dir = os.path.join(DATA_DIR, split_name)
    system_name = "ett1/15T"
    to_univariate, term = False, "long"
    selected_sample_idx, selected_dim = 1, list(range(7))

    dataset = GiftEvalDataset(name=system_name, term=term, to_univariate=to_univariate, data_dir=split_dir)

    # pretty name
    system_name_pretty = f"{system_name.replace('/', '_')}_dim{selected_dim}_sample{selected_sample_idx}"

    prediction_length = dataset.test_data.prediction_length
    print(f"Prediction length: {prediction_length}, Windows: {dataset.windows}")

    # Get the selected sample efficiently
    context_entry = next(islice(dataset.test_data.input, selected_sample_idx, None))
    future_entry = next(islice(dataset.test_data.label, selected_sample_idx, None))
    print(f"Item: {context_entry['item_id']}, Freq: {context_entry['freq']}")

    # Extract and select dimensions
    context_target = context_entry["target"]
    print(f"context_target shape: {context_target.shape}")
    future_target = future_entry["target"]
    if context_target.ndim == 1:
        context_target, future_target = context_target[None, :], future_target[None, :]
    else:
        context_target, future_target = context_target[selected_dim], future_target[selected_dim]

    # Limit context to last DEFAULT_CONTEXT_LENGTH points and adjust start time
    orig_context_len = context_target.shape[1]
    truncate_offset = max(0, orig_context_len - DEFAULT_CONTEXT_LENGTH)
    context_target = context_target[:, -DEFAULT_CONTEXT_LENGTH:]
    context_start = context_entry["start"] + truncate_offset
    context_timesteps = np.arange(orig_context_len)[-DEFAULT_CONTEXT_LENGTH:]
    future_timesteps = np.arange(orig_context_len, orig_context_len + prediction_length)
    print(f"Context: {context_target.shape}, Future: {future_target.shape}")

    # Plot
    num_variates = context_target.shape[0]
    fig, axes = plt.subplots(num_variates, 1, figsize=(5, 2 * num_variates), squeeze=False)
    for dim, ax in enumerate(axes.flat):
        to_pandas({"target": context_target[dim], "start": context_start}).plot(
            color="black", linewidth=1, ax=ax, label="Context"
        )
        to_pandas({"target": future_target[dim], "start": future_entry["start"]}).plot(
            color="tab:blue", linewidth=1, ax=ax, label="Ground Truth"
        )
        ax.grid(which="both")
        ax.set_title(f"Dim {dim}")
        ax.legend()
    plt.tight_layout()
    plt.show()

    # Convert to torch tensors
    context = torch.tensor(context_target)
    future_vals = torch.tensor(future_target)
    num_nans = np.isnan(context_target).sum() + np.isnan(future_target).sum()
    print(f"Context: {context.shape}, Future: {future_vals.shape}, NaNs: {num_nans}")

### Ablations

In [ ]:
import json
from itertools import groupby

ASSETS_DIR = os.path.join("../../", "assets")

heads_to_ablate = json.load(
    open(os.path.join(ASSETS_DIR, f"{pipeline.__class__.__name__}_ablate_for_heads_at_1pp.json"))
)

# chosen_layers = list(range(pipeline.num_layers))
# chosen_layers = [4, 5, 6, 7, 8, 9, 10, 11]
chosen_layers = [2, 9, 10, 11]
print(f"chosen_layers: {chosen_layers}")

heads_to_ablate = [config for config in heads_to_ablate if config[0] in chosen_layers]

num_heads_per_layer_to_skip = 0
layers_to_keep_at_heads1pp = [2, 9, 10, 11]

if num_heads_per_layer_to_skip == 0:
    layers_to_keep_at_heads1pp = chosen_layers

heads_to_ablate = [
    config
    for layer, group in groupby(sorted(heads_to_ablate, key=lambda x: x[0]), key=lambda x: x[0])
    for config in (list(group) if layer in layers_to_keep_at_heads1pp else list(group)[:-num_heads_per_layer_to_skip])
]


In [ ]:
use_random_heads = False

if use_random_heads:
    # now replace heads_to_ablate with randomly chosen heads, in range (0, num_heads), with same number of heads ablated per layer
    rng = np.random.default_rng(321)
    num_heads = num_heads
    heads_to_ablate_random = []
    for layer in chosen_layers:
        heads_in_layer = [config[1] for config in heads_to_ablate if config[0] == layer]
        num_heads_to_ablate = len(heads_in_layer)
        random_heads = rng.choice(range(num_heads), num_heads_to_ablate, replace=False)
        heads_to_ablate_random.extend([(layer, head) for head in random_heads])
    heads_to_ablate = heads_to_ablate_random

In [ ]:
# chosen_layers_mlp = []
chosen_layers_mlp = []
# chosen_layers_mlp = [10]
# chosen_layers_mlp = chosen_layers
print(f"layers chosen for MLP ablation: {chosen_layers_mlp}")

In [ ]:
pipeline.remove_all_hooks()

# print number of heads ablated per layer
for layer in chosen_layers:
    heads_in_layer = [config[1] for config in heads_to_ablate if config[0] == layer]
    print(f"Layer {layer}: {len(heads_in_layer)} heads")

n_heads_ablated = len(heads_to_ablate)
print(f"{n_heads_ablated} heads_to_ablate: {heads_to_ablate}")

ablations_summary_str = f"ablate_{n_heads_ablated}_heads"
if use_random_heads:
    ablations_summary_str = "random_" + ablations_summary_str

print(f"ablations_summary_str: {ablations_summary_str}")

pipeline.add_ablation_hooks_explicit(
    ablations_types=["head", "mlp"],  # type: ignore
    layers_to_ablate_mlp=chosen_layers_mlp,
    heads_to_ablate=heads_to_ablate,
)

layers_without_heads = list(pipeline.head_ablation_handles.keys())
layers_without_mlps = list(pipeline.mlp_ablation_handles.keys())

ablations_summary_str_suffix = ""
if layers_without_heads and layers_without_mlps:
    # if layers_without_heads != layers_without_mlps:
    #     raise NotImplementedError("Zero-ablation of heads and MLPs in different layers is messier, save for later")
    ablations_summary_str_title = (
        f"Zero-Ablation: Heads in Layers {layers_without_heads}, MLPs in Layers {layers_without_mlps}"
    )
    ablations_summary_str_suffix = f"za_heads_layers_{'-'.join(map(str, layers_without_heads))}-mlps_layers_{'-'.join(map(str, layers_without_mlps))}"
elif layers_without_heads:
    ablations_summary_str_title = f"Zero-Ablation: Heads in Layers {layers_without_heads}"
    ablations_summary_str_suffix = f"za_heads_layers_{'-'.join(map(str, layers_without_heads))}"
elif layers_without_mlps:
    ablations_summary_str_title = f"Zero-Ablation: MLPs in Layers {layers_without_mlps}"
    ablations_summary_str_suffix = f"za_mlps_layers_{'-'.join(map(str, layers_without_mlps))}"
else:
    ablations_summary_str_suffix = ablations_summary_str_title = None

ablations_summary_str += "_" + ablations_summary_str_suffix
print(ablations_summary_str)
print(ablations_summary_str_title)


In [ ]:
save_fname = (
    f"predictions_{system_name_pretty}_dim{selected_dim}_start{context_start_time}_sub{subsample_interval}"
    if split_name == "base40"
    else f"predictions_{system_name_pretty}"
)

In [ ]:
# dict(pipeline.model.transformer.layers[7].attention.wO._forward_hooks)

In [ ]:
# pipeline.head_ablation_handles

### Predict and Get Outputs

In [ ]:
context.shape

In [ ]:
rseed = 123
num_samples = 20

set_seed(rseed)

pred = pipeline.predict(
    context,
    prediction_length=prediction_length,
    num_samples=num_samples,  # Number of samples for probabilistic forecasting
    samples_per_batch=num_samples,  # Control memory usage during inference
)

In [ ]:
pred.shape

In [ ]:
pipeline.model.patch_embed.projection.in_features

### Plot Predictions with Ablations

In [ ]:
plot_samples = False

In [ ]:
# Prepare context and predictions
context_np = context.cpu().numpy()
future_vals_np = future_vals
print(f"context_np shape: {context_np.shape}")
print(f"future_vals_np shape: {future_vals_np.shape}")
print(f"length of context_timesteps: {len(context_timesteps)}")
print(f"length of future_timesteps: {len(future_timesteps)}")

In [ ]:
preds = pred.squeeze().detach().cpu().numpy()
print(f"preds shape: {preds.shape}")
if preds.ndim == 2:
    preds = preds[None, ...]

In [ ]:
num_variates, num_samples, prediction_length = preds.shape

save_path = os.path.join(plot_save_dir, f"preds_with_ablations_{ablations_summary_str}.pdf")
print(f"Saving plot to {save_path}")

fig, axs = plt.subplots(num_variates, 1, figsize=(8, 2 * num_variates))
axs = [axs] if num_variates == 1 else axs

for dim, ax in enumerate(axs):
    ax.plot(context_timesteps[-512:], context_np[dim][-512:], "k-", linewidth=1, label="Context", rasterized=True)
    ax.plot(future_timesteps, future_vals_np[dim], "k--", linewidth=1, label="Ground Truth", zorder=10, rasterized=True)
    median_pred = np.median(preds, axis=1)[dim]
    if plot_samples:
        for i in range(num_samples):
            ax.plot(
                future_timesteps, preds[dim, i], linewidth=0.2, color=DEFAULT_COLORS[i], alpha=0.08, rasterized=True
            )
    ax.set_ylim(
        min(context_np[dim][-512:].min(), median_pred.min()), max(context_np[dim][-512:].max(), median_pred.max())
    )
    ax.plot(
        future_timesteps,
        median_pred,
        color="tab:blue",
        linewidth=1,
        label="Median Prediction",
        zorder=100,
        rasterized=True,
    )
    if dim == len(axs) - 1:
        ax.set_xlabel("Timestep", fontweight="bold")
    else:
        ax.set_xticklabels([])
    ax.set_ylabel(f"Dimension {dim}", fontweight="bold")
    if dim == 0:
        ax.legend(loc="upper left", frameon=True)

fig.suptitle(system_name, fontweight="bold", fontsize=12, y=1.0)
# Save plot
save_path = os.path.join(plot_save_dir, "zero_ablations", f"{ablations_summary_str}_{save_fname}.pdf")

if ablations_summary_str_title:
    print(ablations_summary_str_title)
print(f"Saving to {save_path}")
os.makedirs(os.path.dirname(save_path), exist_ok=True)
fig.tight_layout()
fig.savefig(save_path, bbox_inches="tight", dpi=150)


### Plot Original Predictions

In [ ]:
context.shape

In [ ]:
pipeline.remove_all_hooks()
set_seed(rseed)

pred_original = pipeline.predict(
    context,
    prediction_length=prediction_length,
    num_samples=num_samples,  # Number of samples for probabilistic forecasting
    samples_per_batch=num_samples,  # Control memory usage during inference
)

In [ ]:
preds_original = pred_original.squeeze().detach().cpu().numpy()
print(f"preds_original shape: {preds_original.shape}")
if preds_original.ndim == 2:
    preds_original = preds_original[None, ...]

In [ ]:
num_variates, num_samples, prediction_length = preds_original.shape

save_path = os.path.join(plot_save_dir, f"preds_original_{ablations_summary_str}.pdf")
print(f"Saving plot to {save_path}")

fig, axs = plt.subplots(num_variates, 1, figsize=(8, 2 * num_variates))
axs = [axs] if num_variates == 1 else axs

for dim, ax in enumerate(axs):
    ax.plot(context_timesteps[-512:], context_np[dim][-512:], "k-", linewidth=1, label="Context", rasterized=True)
    ax.plot(future_timesteps, future_vals_np[dim], "k--", linewidth=1, label="Ground Truth", zorder=10, rasterized=True)
    median_pred = np.median(preds_original, axis=1)[dim]
    if plot_samples:
        for i in range(num_samples):
            ax.plot(
                future_timesteps,
                preds_original[dim, i],
                linewidth=0.2,
                color=DEFAULT_COLORS[i],
                alpha=0.08,
                rasterized=True,
            )
    ax.set_ylim(
        min(context_np[dim][-512:].min(), median_pred.min()), max(context_np[dim][-512:].max(), median_pred.max())
    )
    ax.plot(
        future_timesteps,
        median_pred,
        color="tab:blue",
        linewidth=1,
        label="Median Prediction",
        zorder=100,
        rasterized=True,
    )
    if dim == len(axs) - 1:
        ax.set_xlabel("Timestep", fontweight="bold")
    else:
        ax.set_xticklabels([])
    ax.set_ylabel(f"Dimension {dim}", fontweight="bold")
    if dim == 0:
        ax.legend(loc="upper left", frameon=True)

fig.suptitle(system_name, fontweight="bold", fontsize=12, y=1.0)
# Save plot
save_path = os.path.join(plot_save_dir, "original", f"{save_fname}.pdf")
print(f"Saving to {save_path}")
os.makedirs(os.path.dirname(save_path), exist_ok=True)
fig.tight_layout()
fig.savefig(save_path, bbox_inches="tight", dpi=150)


### Compute Metrics

In [ ]:
med_pred = np.median(preds, axis=0)
med_orig = np.median(preds_original, axis=0)
print(f"med_pred shape: {med_pred.shape}")
print(f"med_orig shape: {med_orig.shape}")

In [ ]:
pred_horizon_lst = [64]
for pred_horizon in pred_horizon_lst:
    for dim in range(num_variates):
        print(f"Computing metrics for prediction horizon {pred_horizon} and dimension {dim}")
        metrics_combined = compute_metrics(med_orig[dim, :pred_horizon], med_pred[dim, :pred_horizon])
        spearman_corr = metrics_combined["spearman"]
        print(f"Spearman correlation: {spearman_corr}")
        # pprint(metrics_combined, width=1, indent=2)